# Integer Partition Gray Code Constructions

Experiments with deterministic (non-heuristic) Gray code constructions.
Each section is self-contained. Use `verify(listing, n)` to check correctness.

**Operations:**
- *move-one-unit*: transfer 1 unit between two parts (removing 0-parts); or split off a new part of size 1
- *split*: split one part into two smaller parts
- *merge*: merge two parts into one

**Goal:** A *correct* Gray code visits every partition of n exactly once, and each consecutive pair differs by exactly one operation.

## Section 0 — Setup

In [ ]:
# All integer partitions of n as weakly-decreasing tuples.
def allPartitions(n):
    def helper(n, max_part):
        if n == 0:
            yield ()
            return
        for i in range(min(n, max_part), 0, -1):
            for rest in helper(n - i, i):
                yield (i,) + rest
    return list(helper(n, n))

In [ ]:
# Move-one-unit neighbors: transfer 1 unit from part i to part j (removes 0-parts),
# or split off a new part of size 1 from any part of size >= 2.
# Note: the split-off-1 branch overlaps with s=1 splits in neighborsSplitMerge;
# neighborsCombined handles this cleanly via set union.
def neighborsMove(partition):
    parts = list(partition)
    results = set()
    n = sum(parts)
    for i in range(len(parts)):
        for j in range(len(parts)):
            if i == j:
                continue
            new_parts = parts[:]
            new_parts[i] -= 1
            new_parts[j] += 1
            new_parts = [p for p in new_parts if p > 0]
            results.add(tuple(sorted(new_parts, reverse=True)))
        if parts[i] > 1:
            new_parts = parts[:]
            new_parts[i] -= 1
            new_parts.append(1)
            results.add(tuple(sorted(new_parts, reverse=True)))
    return results - {partition}

# Split/merge neighbors: split one part into two, or merge two parts into one.
def neighborsSplitMerge(partition):
    parts = list(partition)
    results = set()
    for i in range(len(parts)):
        for j in range(i + 1, len(parts)):
            merged = parts[i] + parts[j]
            new_parts = [parts[k] for k in range(len(parts)) if k != i and k != j]
            new_parts.append(merged)
            results.add(tuple(sorted(new_parts, reverse=True)))
        for s in range(1, parts[i] // 2 + 1):
            new_parts = parts[:]
            new_parts[i] -= s
            new_parts.append(s)
            results.add(tuple(sorted(new_parts, reverse=True)))
    return results - {partition}

# Combined neighborhood: union of move-one-unit and split/merge.
def neighborsCombined(partition):
    return neighborsMove(partition) | neighborsSplitMerge(partition)

In [ ]:
# Encode a partition as a length-n binary string (LAGOS 2025 paper encoding).
# Each part a_k -> 0^(a_k-1) + '1', all parts concatenated.
# Example: (3,2,1) -> "001" + "01" + "1" = "001011"
def toBinary(partition):
    return ''.join('0' * (p - 1) + '1' for p in partition)

In [ ]:
# Verify a listing is a correct Gray code for integer partitions of n.
# Returns (True, "OK") or (False, reason string).
def verify(listing, n):
    expected = allPartitions(n)
    if len(listing) != len(expected):
        return False, f"Length {len(listing)}, expected {len(expected)}"
    if set(listing) != set(expected):
        missing = set(expected) - set(listing)
        return False, f"Missing partitions: {sorted(missing)[:5]}"
    for i in range(len(listing) - 1):
        if listing[i + 1] not in neighborsCombined(listing[i]):
            return False, f"Step {i}: {listing[i]} -> {listing[i+1]} is not a valid neighbor"
    return True, "OK"

In [ ]:
# Smoke test: verify utilities on n=5.
parts5 = allPartitions(5)
print(f"P(5) = {len(parts5)} partitions: {parts5}")
print(f"neighborsMove((3,2)):    {sorted(neighborsMove((3,2)))}")
print(f"neighborsSplitMerge((3,2)): {sorted(neighborsSplitMerge((3,2)))}")
print(f"toBinary((3,2)) = '{toBinary((3,2))}'  (expected '00101')")